In [1]:
import pandas as pd
import numpy as np
import re
import json
import joblib
import os
from tqdm import tqdm
tqdm.pandas()

# NLP
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

h:\Resume shortlister\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
df = pd.read_csv("../data/processed/cleaned_resumes.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label distribution:\n", df['Decision'].value_counts())
df.head(3)

Shape: (5323, 6)
Columns: ['Role', 'Resume', 'Decision', 'Reason_for_decision', 'Job_Description', 'Resume_clean']
Label distribution:
 Decision
reject    2721
select    2602
Name: count, dtype: int64


,Role,Resume,Decision,Reason_for_decision,Job_Description,Resume_clean
0,Mobile App Developer,"Here's a sample resume for Jose Hall, a skille...",reject,No experience in back-end development.,We need a Mobile App Developer to enhance our ...,Professional Summary: Highly motivated and det...
1,Cloud Engineer,"Here's a sample resume for Jessica Hall, a can...",reject,Needs improvement in machine learning algorithms.,We're seeking a talented Cloud Engineer to wor...,Summary: Highly motivated and detail-oriented ...
2,Ai Researcher,Here's a sample resume for Zachary Ward:\n\nZa...,select,Excellent full-stack development experience.,If you're passionate about software engineerin...,Professional Summary: Highly motivated AI rese...


In [3]:
# Load spaCy
print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")

# Load MiniLM sentence transformer
print("Loading MiniLM embeddings model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Both models loaded")

Loading spaCy...
Loading MiniLM embeddings model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1532.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Both models loaded


In [ ]:
# Master skill list — covers all roles in your dataset
TECH_SKILLS = [
    # ML / AI
    'python', 'machine learning', 'deep learning', 'tensorflow',
    'pytorch', 'scikit-learn', 'xgboost', 'keras', 'nlp',
    'computer vision', 'reinforcement learning', 'mlflow',
    'hugging face', 'transformers', 'langchain', 'llm',
    'rag', 'faiss', 'openai', 'prompt engineering',
    'feature engineering', 'shap', 'optuna', 'lightgbm',

    # Data
    'pandas', 'numpy', 'sql', 'spark', 'kafka', 'airflow',
    'dbt', 'bigquery', 'snowflake', 'hadoop', 'pyspark',
    'etl', 'data warehouse', 'tableau', 'power bi',
    'matplotlib', 'seaborn', 'statistics',

    # Engineering
    'docker', 'kubernetes', 'aws', 'azure', 'gcp',
    'google cloud', 'git', 'linux', 'fastapi', 'flask',
    'django', 'rest api', 'postgresql', 'mongodb',
    'redis', 'ci/cd', 'jenkins', 'terraform',

    # Mobile / Full Stack
    'react', 'nodejs', 'javascript', 'typescript',
    'swift', 'kotlin', 'flutter', 'android', 'ios',

    # Security / Other
    'cybersecurity', 'penetration testing', 'blockchain',
    'solidity', 'unity', 'unreal engine','hive', 'scala', 'matlab', 'r',
    'power automate', 'looker', 'grafana',
    'prometheus', 'elasticsearch', 'neo4j'
]

print(f"Total skills in master list: {len(TECH_SKILLS)}")

Total skills in master list: 75


In [5]:
def extract_skills_spacy(text):
    """
    Extract skills using spaCy noun chunks
    matched against our master skill list
    """
    if not isinstance(text, str):
        return []

    text_lower = text.lower()
    doc = nlp(text_lower)

    found_skills = []

    # Method 1 — Direct keyword matching
    for skill in TECH_SKILLS:
        if skill in text_lower:
            found_skills.append(skill)

    # Method 2 — spaCy noun chunks for additional extraction
    noun_chunks = [chunk.text.lower().strip() for chunk in doc.noun_chunks]
    for chunk in noun_chunks:
        if chunk in TECH_SKILLS and chunk not in found_skills:
            found_skills.append(chunk)

    return list(set(found_skills))


# Test it
sample_text = df['Resume_clean'].iloc[0]
skills_found = extract_skills_spacy(sample_text)
print(f"Skills found in sample resume:")
print(skills_found)
print(f"Total: {len(skills_found)} skills")

Skills found in sample resume:
['swift', 'ios', 'git', 'flutter', 'android', 'kotlin']
Total: 6 skills


In [6]:
# Check a Machine Learning Engineer resume specifically
ml_resumes = df[df['Role'] == 'Machine Learning Engineer']
print(f"Total ML Engineer resumes: {len(ml_resumes)}")

# Get index of first ML resume
ml_idx = ml_resumes.index[0]

# Extract skills
sample_text = df['Resume_clean'].iloc[ml_idx]
skills_found = extract_skills_spacy(sample_text)

print(f"\nRole: {df['Role'].iloc[ml_idx]}")
print(f"\nSkills found:")
print(skills_found)
print(f"\nTotal: {len(skills_found)} skills")
print(f"\nResume preview:")
print(sample_text[:300])

Total ML Engineer resumes: 265

Role: Machine Learning Engineer

Skills found:
['keras', 'tensorflow', 'sql', 'machine learning', 'deep learning', 'computer vision', 'python', 'rag', 'numpy', 'linux', 'scikit-learn', 'pandas', 'pytorch']

Total: 13 skills

Resume preview:
Professional Summary: Highly motivated and detail-oriented Machine Learning Engineer with 5+ years of experience in developing and deploying scalable, accurate, and efficient machine learning models using Python, TensorFlow, and PyTorch. Proven track record of delivering projects on time and exceedi


In [9]:
def extract_skills_spacy(text):
    if not isinstance(text, str):
        return []
    
    text_lower = text.lower()
    doc = nlp(text_lower)
    found_skills = []

    # Method 1 — Fixed skill list matching
    for skill in TECH_SKILLS:
        if skill in text_lower:
            found_skills.append(skill)

    # Method 2 — Dynamic noun extraction
    # catches skills NOT in our fixed list
    for token in doc:
        if token.pos_ in ['NOUN', 'PROPN']:
            word = token.text.lower().strip()
            if (len(word) > 2 and
                word not in found_skills and
                word.isalpha()):
                found_skills.append(word)

    return list(set(found_skills))

In [ ]:
print("Extracting skills from resumes...")
df['resume_skills'] = df['Resume_clean'].progress_apply(extract_skills_spacy)

print("Extracting skills from JDs...")
df['jd_skills'] = df['Job_Description'].progress_apply(extract_skills_spacy)

print("✅ Done")

# Verify
print(f"\nAvg skills per resume : {df['resume_skills'].apply(len).mean():.1f}")
print(f"Avg skills per JD     : {df['jd_skills'].apply(len).mean():.1f}")
print(f"Avg overlap           : {df.apply(lambda r: len(set(r['resume_skills']) & set(r['jd_skills'])), axis=1).mean():.1f}")
print(f"\nJD skills sample: {df['jd_skills'].iloc[0]}")

JD skills sample:
[]

Avg skills per resume : 10.1
Avg skills per JD     : 1.5
Avg overlap count     : 0.9
